### InMemoryVectorStore
In-memory vector store implementation.

Uses a dictionary, and computes cosine similarity for search using numpy.

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
from langchain_classic.chat_models import init_chat_model

llm=init_chat_model('groq:groq:llama-3.3-70b-versatile')
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x159b1ef90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x15bb00590>, model_name='groq:llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = InMemoryVectorStore(embedding=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7172.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [11]:
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [13]:
vector_store.add_documents(documents=documents)

['0e9a1243-b7c5-4603-9ee2-2bdd9fbbff8b',
 '1337f70a-6620-4823-bb5e-b8452451ffb5',
 '103f18c1-0afc-4806-809a-4895413e01d6',
 'aa0b715c-332a-4648-abe1-f020cca53540',
 '7815aeb7-8d21-40f8-8526-32ca154862ff',
 'f57c5d7d-4f9e-4001-8a87-2340fd6725fd',
 'be38ab63-755e-4430-970a-98a7becc8967',
 '86b25ffb-25ce-4b46-b9d7-9ce037bf373e',
 'a8e71c9f-6514-406f-8446-f04d08f7b46c',
 '6a532e54-6a78-4053-bfb1-933bd8a4b99d']

In [14]:
vector_store.similarity_search("hows the weather forecast")

[Document(id='1337f70a-6620-4823-bb5e-b8452451ffb5', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='a8e71c9f-6514-406f-8446-f04d08f7b46c', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='86b25ffb-25ce-4b46-b9d7-9ce037bf373e', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='6a532e54-6a78-4053-bfb1-933bd8a4b99d', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [15]:
vector_store.similarity_search("hows the weather forecast",k=2)

[Document(id='1337f70a-6620-4823-bb5e-b8452451ffb5', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='a8e71c9f-6514-406f-8446-f04d08f7b46c', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]

In [16]:
### vectorstore to retriever

retriever=vector_store.as_retriever(search_kwargs={"k":2})

retriever

VectorStoreRetriever(tags=['InMemoryVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x1599c3a80>, search_kwargs={'k': 2})

In [17]:
## Invoke
retriever.invoke("hows the weather forecast")

[Document(id='1337f70a-6620-4823-bb5e-b8452451ffb5', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='a8e71c9f-6514-406f-8446-f04d08f7b46c', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]